In [1]:
import os
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from glob import glob
import numpy as np

import rasterio
from rasterio.features import rasterize

from shapely import wkt
from shapely.geometry import box
from shapely.geometry import mapping

In [2]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')
flights = '/store/carroll/col/2018/raw/rmbl/' # still only 4 / 7 days, will rerun once have everything

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'pixel'

In [3]:
# load and view relevant schema and dtype for the table
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# fix typos
schema['column_name'] = schema['column_name'].replace('wavelenght_center', 'wavelength_center')

schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
38,pixel,plot_name,character
39,pixel,glt_row,integer
40,pixel,glt_column,integer
41,pixel,flightline_id,uuid
42,pixel,pixel_id,uuid
43,pixel,scene_id,uuid


In [4]:
# load relevant output tables

flightline = pd.read_csv(os.path.join(out_folder, 'flightline.csv'))

raster_plot_event = pd.read_csv(os.path.join(out_folder, 'raster_plot_event.csv'))
raster_plot_event['geometry'] = raster_plot_event['geom'].apply(wkt.loads)
raster_plot_event = gpd.GeoDataFrame(raster_plot_event, geometry=raster_plot_event.geometry, crs=32613)

In [5]:
# for each flightline, use raster_plot_event polygons to extract the relevant pixels (only those contained entirely within, and not-NA)
# also populate flightline_id and glt row, col
# use loc data for this because smallest files

flightlines = glob('/store/carroll/col/2018/raw/rmbl/*/*_igm_ort')
len(flightlines)

59

In [ ]:
out = []

for fp in flightlines:
    flight = fp.split('/')[-1].removesuffix('_rdn_ort_igm_ort')
    print(flight)
    with rasterio.open(fp) as src:
        # filter polygons to those fully covered by raster extent
        extent_ = box(*src.bounds)
        tmp = raster_plot_event[raster_plot_event.geometry.covered_by(extent_)]
        # rasterize filtered gdf
        shapes = [(mapping(geom), val) for geom, val in zip(tmp.geometry, tmp.index)]
        r = rasterize(shapes, out_shape=(src.height, src.width), transform=src.transform, all_touched=False, nodata=-9999, fill=-9999)
        # nodata mask
        nodata = src.read(1)<=0
    r[nodata] = -9999
    # extract glt_row, glt_col, plot_name
    mask = r!=-9999
    row, col = np.where(mask)
    val = r[mask]
    plot = [tmp['plot_name'][x] for x in val]
    df = pd.DataFrame({
        'glt_row': row,
        'glt_column': col,
        'plot_name': plot,
        'flightline_id': flight
    })
    out.append(df)

df = pd.concat(out)

In [10]:
# prepare & populate out table
out_table = df

out_table['pixel_id'] = ['px'+str(x) for x in range(len(out_table))] # temporarily populate
out_table['scene_id'] = pd.NA

# reorder columns
out_table = out_table[schema.column_name]

out_table

,plot_name,glt_row,glt_column,flightline_id,pixel_id,scene_id
0,056-ER18,287,51,NIS01_20180612_175319,px0,<NA>
1,056-ER18,287,52,NIS01_20180612_175319,px1,<NA>
2,056-ER18,288,51,NIS01_20180612_175319,px2,<NA>
3,056-ER18,288,52,NIS01_20180612_175319,px3,<NA>
0,056-ER18,10552,554,NIS01_20180613_154908,px4,<NA>
...,...,...,...,...,...,...
104,205-ER18,16984,135,NIS01_20180620_173209,px5471,<NA>
105,197-ER18,16992,143,NIS01_20180620_173209,px5472,<NA>
106,197-ER18,16992,144,NIS01_20180620_173209,px5473,<NA>
107,204-ER18,17004,118,NIS01_20180620_173209,px5474,<NA>


In [11]:
# confirm final column data types

print(out_table.dtypes)

out_table.dtypes

plot_name        object
glt_row           int64
glt_column        int64
flightline_id    object
pixel_id         object
scene_id         object
dtype: object


plot_name        object
glt_row           int64
glt_column        int64
flightline_id    object
pixel_id         object
scene_id         object
dtype: object

In [12]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)

In [ ]:
# again, will need to re-run thtis once I have all of the flghtlines, which will make raster_plot_event larger. But utility is set up